# **Live Coding 1**

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, avg
import time

1. Crear SparkSession

In [12]:
spark = SparkSession.builder \
    .appName("Demo Spark SQL") \
    .getOrCreate()

print("SparkSession creada correctamente")
print("Versión de Spark:", spark.version)

SparkSession creada correctamente
Versión de Spark: 4.0.2


2. Cargar datos desde CSV

In [13]:
ventas_df = spark.read.csv(
    "ventas.csv",
    header=True,
    inferSchema=True
)

print("\nSchema de ventas:")
ventas_df.printSchema()

print("\nPrimeras filas de ventas:")
ventas_df.show()



Schema de ventas:
root
 |-- id_venta: integer (nullable = true)
 |-- id_sucursal: integer (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- monto: integer (nullable = true)
 |-- fecha: date (nullable = true)


Primeras filas de ventas:
+--------+-----------+----------+----------+------+----------+
|id_venta|id_sucursal|  producto| categoria| monto|     fecha|
+--------+-----------+----------+----------+------+----------+
|       1|        784|   Mochila|Accesorios|438201|2025-05-04|
|       2|        144|     Mouse|Tecnologia|266498|2025-09-23|
|       3|        313|   Monitor|Tecnologia|426047|2025-10-27|
|       4|        837|   Teclado|Tecnologia|523122|2025-11-02|
|       5|        673| Impresora|   Oficina|845609|2025-02-11|
|       6|        846|   Mochila|Accesorios|719403|2025-05-15|
|       7|        635|   Monitor|Tecnologia|828197|2025-06-04|
|       8|        823|Escritorio|   Oficina|915489|2025-05-17|
|       9|      

In [15]:
ventas_df.select("categoria").distinct().show()

+----------+
| categoria|
+----------+
|    Utiles|
|Accesorios|
|   Oficina|
|Tecnologia|
+----------+



3. Cargar datos desde JSON

In [16]:
sucursales_df = spark.read.json("sucursales.json")

print("\nSchema de sucursales:")
sucursales_df.printSchema()

print("\nPrimeras filas de sucursales:")
sucursales_df.show()



Schema de sucursales:
root
 |-- id_sucursal: long (nullable = true)
 |-- nombre_sucursal: string (nullable = true)
 |-- region: string (nullable = true)


Primeras filas de sucursales:
+-----------+---------------+-------------+
|id_sucursal|nombre_sucursal|       region|
+-----------+---------------+-------------+
|        101|   Sucursal 101|    Los Lagos|
|        102|   Sucursal 102|   Valparaiso|
|        103|   Sucursal 103|   Valparaiso|
|        104|   Sucursal 104|    Los Lagos|
|        105|   Sucursal 105|     Coquimbo|
|        106|   Sucursal 106|   Valparaiso|
|        107|   Sucursal 107|        Maule|
|        108|   Sucursal 108|    Araucania|
|        109|   Sucursal 109|  Antofagasta|
|        110|   Sucursal 110|    Araucania|
|        111|   Sucursal 111|    Araucania|
|        112|   Sucursal 112|  Antofagasta|
|        113|   Sucursal 113|Metropolitana|
|        114|   Sucursal 114|    Araucania|
|        115|   Sucursal 115|    Araucania|
|        116|   Sucurs

In [18]:
sucursales_df.select("region").distinct().show()

+-------------+
|       region|
+-------------+
|     Coquimbo|
|       Biobio|
|        Maule|
|    Los Lagos|
|   Valparaiso|
|  Antofagasta|
|Metropolitana|
|    Araucania|
+-------------+



4. Join entre DataFrames

In [5]:
df_join = ventas_df.join(sucursales_df, on="id_sucursal", how="inner")

print("\nDataFrame unido:")
df_join.show()


DataFrame unido:
+-----------+--------+----------+----------+------+----------+---------------+-------------+
|id_sucursal|id_venta|  producto| categoria| monto|     fecha|nombre_sucursal|       region|
+-----------+--------+----------+----------+------+----------+---------------+-------------+
|        784|       1|   Mochila|Accesorios|438201|2025-05-04|   Sucursal 784|    Los Lagos|
|        144|       2|     Mouse|Tecnologia|266498|2025-09-23|   Sucursal 144|    Los Lagos|
|        313|       3|   Monitor|Tecnologia|426047|2025-10-27|   Sucursal 313|    Araucania|
|        837|       4|   Teclado|Tecnologia|523122|2025-11-02|   Sucursal 837|Metropolitana|
|        673|       5| Impresora|   Oficina|845609|2025-02-11|   Sucursal 673|   Valparaiso|
|        846|       6|   Mochila|Accesorios|719403|2025-05-15|   Sucursal 846|        Maule|
|        635|       7|   Monitor|Tecnologia|828197|2025-06-04|   Sucursal 635|    Araucania|
|        823|       8|Escritorio|   Oficina|915489|2

5. Registrar vista temporal

In [6]:
df_join.createOrReplaceTempView("ventas_sucursales")

6. Consulta SQL con filtros y agregaciones

In [7]:
consulta = """
SELECT
    region,
    categoria,
    COUNT(*) AS cantidad_ventas,
    SUM(monto) AS total_ventas,
    ROUND(AVG(monto), 2) AS promedio_venta
FROM ventas_sucursales
WHERE monto > 10000
GROUP BY region, categoria
ORDER BY total_ventas DESC
"""

resultado_sql = spark.sql(consulta)

print("\nResultado de la consulta SQL:")
resultado_sql.show()


Resultado de la consulta SQL:
+-------------+----------+---------------+------------+--------------+
|       region| categoria|cantidad_ventas|total_ventas|promedio_venta|
+-------------+----------+---------------+------------+--------------+
|        Maule|Tecnologia|          57344| 27557990767|     480573.22|
|    Araucania|Tecnologia|          53729| 25746748299|     479196.49|
|    Los Lagos|Tecnologia|          52050| 24925091180|     478868.23|
|  Antofagasta|Tecnologia|          48789| 23377169278|     479148.36|
|     Coquimbo|Tecnologia|          46953| 22719304099|     483873.32|
|Metropolitana|Tecnologia|          47170| 22608614781|     479300.72|
|       Biobio|Tecnologia|          46716| 22388645824|     479250.06|
|   Valparaiso|Tecnologia|          43980| 21091986853|     479581.33|
|        Maule|   Oficina|          43175| 20761127233|     480859.92|
|    Araucania|   Oficina|          40557| 19538177668|     481746.13|
|    Los Lagos|   Oficina|          39198| 188

7. Ver plan de ejecución

In [ ]:
print("\nPlan de ejecución:")
resultado_sql.explain(True)


Plan de ejecución:
== Parsed Logical Plan ==
'Sort ['total_ventas DESC NULLS LAST], true
+- 'Aggregate ['region, 'categoria], ['region, 'categoria, 'COUNT(1) AS cantidad_ventas#106, 'SUM('monto) AS total_ventas#107, 'ROUND('AVG('monto), 2) AS promedio_venta#108]
   +- 'Filter ('monto > 10000)
      +- 'UnresolvedRelation [ventas_sucursales], [], false

== Analyzed Logical Plan ==
region: string, categoria: string, cantidad_ventas: bigint, total_ventas: bigint, promedio_venta: double
Sort [total_ventas#107L DESC NULLS LAST], true
+- Aggregate [region#57, categoria#20], [region#57, categoria#20, count(1) AS cantidad_ventas#106L, sum(monto#21) AS total_ventas#107L, round(avg(monto#21), 2) AS promedio_venta#108]
   +- Filter (monto#21 > 10000)
      +- SubqueryAlias ventas_sucursales
         +- View (`ventas_sucursales`, [id_sucursal#18, id_venta#17, producto#19, categoria#20, monto#21, fecha#22, nombre_sucursal#56, region#57])
            +- Project [id_sucursal#18, id_venta#17, product

8. Mini comparación de rendimiento

In [8]:
consulta = """
SELECT
    region,
    categoria,
    COUNT(*) AS cantidad_ventas,
    SUM(monto) AS total_ventas,
    ROUND(AVG(monto), 2) AS promedio_venta
FROM ventas_grandes
WHERE monto > 100000
GROUP BY region, categoria
ORDER BY total_ventas DESC
"""

# Asegurar que la vista exista
df_join.createOrReplaceTempView("ventas_grandes")

# =========================
# 1. Escenario optimizado
# =========================
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.codegen.wholeStage", "true")
spark.conf.set("spark.sql.cbo.enabled", "true")

inicio = time.time()
resultado_opt = spark.sql(consulta)
resultado_opt.show()
fin = time.time()

tiempo_opt = fin - inicio
print(f"Tiempo con optimizaciones activadas: {tiempo_opt:.4f} segundos")

print("\nPlan con optimizaciones activadas:")
resultado_opt.explain(True)



+-------------+----------+---------------+------------+--------------+
|       region| categoria|cantidad_ventas|total_ventas|promedio_venta|
+-------------+----------+---------------+------------+--------------+
|        Maule|Tecnologia|          51832| 27257087382|     525873.73|
|    Araucania|Tecnologia|          48518| 25459278444|     524738.83|
|    Los Lagos|Tecnologia|          47058| 24647579467|     523770.23|
|  Antofagasta|Tecnologia|          44057| 23118664854|     524744.42|
|     Coquimbo|Tecnologia|          42471| 22470951588|      529089.3|
|Metropolitana|Tecnologia|          42610| 22355044971|     524643.16|
|       Biobio|Tecnologia|          42234| 22142225135|     524274.88|
|   Valparaiso|Tecnologia|          39711| 20857602791|     525234.89|
|        Maule|   Oficina|          39038| 20530259695|      525904.5|
|    Araucania|   Oficina|          36734| 19326737329|     526126.68|
|    Los Lagos|   Oficina|          35526| 18659428756|     525233.03|
|  Ant

In [9]:
# =========================
# 2. Escenario menos optimizado
# =========================
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.codegen.wholeStage", "false")
spark.conf.set("spark.sql.cbo.enabled", "false")

inicio = time.time()
resultado_no_opt = spark.sql(consulta)
resultado_no_opt.show()
fin = time.time()

tiempo_no_opt = fin - inicio
print(f"Tiempo con optimizaciones reducidas: {tiempo_no_opt:.4f} segundos")

print("\nPlan con optimizaciones reducidas:")
resultado_no_opt.explain(True)

# =========================
# 3. Comparación final
# =========================
print("\nComparación de tiempos:")
print(f"Con optimizaciones   : {tiempo_opt:.4f} s")
print(f"Sin optimizaciones   : {tiempo_no_opt:.4f} s")

if tiempo_no_opt > 0:
    mejora = ((tiempo_no_opt - tiempo_opt) / tiempo_no_opt) * 100
    print(f"Mejora estimada      : {mejora:.2f}%")

+-------------+----------+---------------+------------+--------------+
|       region| categoria|cantidad_ventas|total_ventas|promedio_venta|
+-------------+----------+---------------+------------+--------------+
|        Maule|Tecnologia|          51832| 27257087382|     525873.73|
|    Araucania|Tecnologia|          48518| 25459278444|     524738.83|
|    Los Lagos|Tecnologia|          47058| 24647579467|     523770.23|
|  Antofagasta|Tecnologia|          44057| 23118664854|     524744.42|
|     Coquimbo|Tecnologia|          42471| 22470951588|      529089.3|
|Metropolitana|Tecnologia|          42610| 22355044971|     524643.16|
|       Biobio|Tecnologia|          42234| 22142225135|     524274.88|
|   Valparaiso|Tecnologia|          39711| 20857602791|     525234.89|
|        Maule|   Oficina|          39038| 20530259695|      525904.5|
|    Araucania|   Oficina|          36734| 19326737329|     526126.68|
|    Los Lagos|   Oficina|          35526| 18659428756|     525233.03|
|  Ant

9. Cerrar sesión


In [10]:
spark.stop()
print("\nSparkSession finalizada")


SparkSession finalizada
